In [ ]:
import pandas as pd
import numpy as np

# Read the CSV file
input_file = r"C:\Users\morte\Downloads\FPMA-main\wheat\wheat_interpolated.csv"
output_file = r"C:\Users\morte\Downloads\FPMA-main\wheat\wheat_interpolated_with_national_avg.csv"

print("Reading CSV file...")
df = pd.read_csv(input_file)

print(f"Original data shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# Filter for Domestic price_source only
df_domestic = df[df['price_source'] == 'Domestic'].copy()
print(f"\nFiltered to Domestic price_source: {df_domestic.shape[0]} rows")

# Define grouping columns
group_cols = [
    "commodity_type",
    "commodity_name",
    "price_source",
    "country",
    "date"  # Added date to ensure we're grouping by time period
]

print("\nProcessing national averages...")

def calculate_national_average(group_df):
    """
    Calculate national average price per unit for a given group.
    
    Guidelines:
    1. For "National Average" market with only Wholesale OR Retail: use that price directly
    2. For "National Average" market with both Wholesale AND Retail: average them first, 
       then average with other markets
    3. If no "National Average" market: average all markets (averaging Wholesale/Retail 
       within each market first)
    """
    
    # Check if there's a National Average market
    nat_avg_df = group_df[group_df['market'] == 'National Average']
    
    if len(nat_avg_df) > 0:
        # Case 1 & 2: National Average market exists
        
        # Get unique price types in National Average market
        nat_avg_price_types = nat_avg_df['price_type'].unique()
        
        if len(nat_avg_price_types) == 1:
            # Case 1: Only one price type (Wholesale OR Retail)
            # Use National Average price directly without averaging with other markets
            nat_avg_price = nat_avg_df['price_per_unit'].mean()
            return nat_avg_price
        
        else:
            # Case 2: Both Wholesale and Retail in National Average
            # First average the National Average market prices
            nat_avg_price = nat_avg_df['price_per_unit'].mean()
            
            # Get other markets (excluding National Average)
            other_markets_df = group_df[group_df['market'] != 'National Average']
            
            if len(other_markets_df) == 0:
                # Only National Average market exists
                return nat_avg_price
            
            # For other markets, average Wholesale/Retail within each market first
            other_markets_avg = []
            for market_name in other_markets_df['market'].unique():
                market_df = other_markets_df[other_markets_df['market'] == market_name]
                market_avg = market_df['price_per_unit'].mean()
                other_markets_avg.append(market_avg)
            
            # Combine National Average with other markets averages
            all_averages = [nat_avg_price] + other_markets_avg
            final_avg = np.mean(all_averages)
            return final_avg
    
    else:
        # Case 3: No National Average market
        # Average all markets, but first average Wholesale/Retail within each market
        
        market_averages = []
        for market_name in group_df['market'].unique():
            market_df = group_df[group_df['market'] == market_name]
            # Average Wholesale and Retail for this market
            market_avg = market_df['price_per_unit'].mean()
            market_averages.append(market_avg)
        
        # Return average of all market averages
        return np.mean(market_averages)


# Group by the specified columns and calculate national average
national_averages = []

total_groups = len(df_domestic.groupby(group_cols))
print(f"Total groups to process: {total_groups}")

for i, (group_keys, group_df) in enumerate(df_domestic.groupby(group_cols)):
    if (i + 1) % 1000 == 0:
        print(f"  Processed {i + 1}/{total_groups} groups...")
    
    nat_avg = calculate_national_average(group_df)
    
    # Create a record for each row in the group with the calculated national average
    for idx in group_df.index:
        national_averages.append({
            'index': idx,
            'national_average_price_per_unit': nat_avg
        })

print(f"Calculated {len(national_averages)} national average values")

# Create a DataFrame from the results
nat_avg_df = pd.DataFrame(national_averages)
nat_avg_df = nat_avg_df.set_index('index')

# Add the national_average_price_per_unit column to the original dataframe
df['national_average_price_per_unit'] = np.nan
df.loc[nat_avg_df.index, 'national_average_price_per_unit'] = nat_avg_df['national_average_price_per_unit']

print("\nNational average column added!")
print(f"Non-null national averages: {df['national_average_price_per_unit'].notna().sum()}")
print(f"Null national averages (non-Domestic or missing data): {df['national_average_price_per_unit'].isna().sum()}")

# Save to new CSV file
print(f"\nSaving to: {output_file}")
df.to_csv(output_file, index=False)

print("Done! File saved successfully.")

# Show some sample results
print("\n=== Sample Results ===")
sample = df[df['national_average_price_per_unit'].notna()].head(10)
print(sample[['date', 'country', 'market', 'commodity_name', 'price_type', 
              'price_per_unit', 'national_average_price_per_unit']])

# Show statistics
print("\n=== Statistics ===")
print(f"Original price_per_unit - Mean: {df['price_per_unit'].mean():.4f}, Std: {df['price_per_unit'].std():.4f}")
print(f"National average - Mean: {df['national_average_price_per_unit'].mean():.4f}, Std: {df['national_average_price_per_unit'].std():.4f}")


Reading CSV file...
Original data shape: (80680, 20)
Columns: ['date', 'price_usd', 'commodity_name', 'commodity_type', 'country', 'iso3', 'region', 'subregion', 'market', 'price_type', 'unit', 'unit_std', 'price_per_unit', 'price_source', 'gdp_ppp17', 'gdppc_ppp', 'gnipc_ppp', 'population', 'inflation', 'income_level']

Filtered to Domestic price_source: 77723 rows

Processing national averages...
Total groups to process: 31559
  Processed 1000/31559 groups...
  Processed 2000/31559 groups...
  Processed 3000/31559 groups...
  Processed 4000/31559 groups...
  Processed 5000/31559 groups...
  Processed 6000/31559 groups...
  Processed 7000/31559 groups...
  Processed 8000/31559 groups...
  Processed 9000/31559 groups...
  Processed 10000/31559 groups...
  Processed 11000/31559 groups...
  Processed 12000/31559 groups...
  Processed 13000/31559 groups...
  Processed 14000/31559 groups...
  Processed 15000/31559 groups...
  Processed 16000/31559 groups...
  Processed 17000/31559 groups..

In [ ]:
import pandas as pd
import numpy as np

# Read the CSV file
input_file = r"C:\Users\morte\Downloads\FPMA-main\wheat\wheat_interpolated.csv"
output_file = r"C:\Users\morte\Downloads\FPMA-main\wheat\wheat_interpolated_with_national_avg.csv"

print("Reading CSV file...")
df = pd.read_csv(input_file)

print(f"Original data shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# Filter for Domestic price_source only
df_domestic = df[df['price_source'] == 'Domestic'].copy()
print(f"\nFiltered to Domestic price_source: {df_domestic.shape[0]} rows")

# Define grouping columns
group_cols = [
    "commodity_type",
    "commodity_name",
    "price_source",
    "country",
    "unit_std",  # Important: don't average prices with different units
    "date"  # Added date to ensure we're grouping by time period
]

print("\nProcessing national averages...")

def calculate_national_average(group_df):
    """
    Calculate national average price per unit for a given group.
    
    Guidelines:
    1. For "National Average" market with only Wholesale OR Retail: use that price directly, 
       IGNORE other markets
    2. For "National Average" market with both Wholesale AND Retail: average them, 
       IGNORE other markets
    3. If no "National Average" market: average all markets (averaging Wholesale/Retail 
       within each market first if both exist)
    """
    
    # Check if there's a National Average market
    nat_avg_df = group_df[group_df['market'] == 'National Average']
    
    if len(nat_avg_df) > 0:
        # Case 1 & 2: National Average market exists
        
        # Get unique price types in National Average market
        nat_avg_price_types = nat_avg_df['price_type'].unique()
        
        if len(nat_avg_price_types) == 1:
            # Case 1: Only one price type (Wholesale OR Retail)
            # Use National Average price directly without averaging with other markets
            nat_avg_price = nat_avg_df['price_per_unit'].mean()
            return nat_avg_price
        
        else:
            # Case 2: Both Wholesale and Retail in National Average
            # Average the National Average market prices and IGNORE other markets
            nat_avg_price = nat_avg_df['price_per_unit'].mean()
            return nat_avg_price
    
    else:
        # Case 3: No National Average market
        # Average all markets, but first average Wholesale/Retail within each market
        
        market_averages = []
        for market_name in group_df['market'].unique():
            market_df = group_df[group_df['market'] == market_name]
            # Average Wholesale and Retail for this market
            market_avg = market_df['price_per_unit'].mean()
            market_averages.append(market_avg)
        
        # Return average of all market averages
        return np.mean(market_averages)


# Initialize - no need for the extra column anymore
# The calculated national average will be in price_per_unit for "Calculated National Average" rows

# Lists to store new rows that need to be created
new_rows = []

total_groups = len(df_domestic.groupby(group_cols))
print(f"Total groups to process: {total_groups}")

for i, (group_keys, group_df) in enumerate(df_domestic.groupby(group_cols)):
    if (i + 1) % 1000 == 0:
        print(f"  Processed {i + 1}/{total_groups} groups...")
    
    # Calculate the national average for this group
    nat_avg = calculate_national_average(group_df)
    
    # Always create a new "Calculated National Average" row for every group
    # Use the first row of the group as a template
    template_row = group_df.iloc[0].copy()
    
    # Modify the template to be a Calculated National Average row
    template_row['market'] = 'Calculated National Average'
    template_row['price_per_unit'] = nat_avg  # The calculated national average goes here
    template_row['price_usd'] = np.nan  # We don't have the original unit price
    template_row['price_type'] = 'Calculated National Average'  # Mark as calculated national average
    
    new_rows.append(template_row)

print(f"\nCreating {len(new_rows)} new Calculated National Average rows...")

# Add the new rows to the dataframe
if len(new_rows) > 0:
    new_rows_df = pd.DataFrame(new_rows)
    df = pd.concat([df, new_rows_df], ignore_index=True)
    print(f"Added {len(new_rows)} new Calculated National Average rows")

print("\nCalculated National Average rows created!")
print(f"Total rows in final dataset: {len(df)}")

# Sort the dataframe for better readability
df = df.sort_values(['country', 'commodity_name', 'date', 'market']).reset_index(drop=True)

# Save to new CSV file
print(f"\nSaving to: {output_file}")
df.to_csv(output_file, index=False)

print("Done! File saved successfully.")

# Show some sample results
print("\n=== Sample Results (Calculated National Average rows only) ===")
sample = df[df['market'] == 'Calculated National Average'].head(15)
print(sample[['date', 'country', 'market', 'commodity_name', 'price_type', 'price_per_unit']])

# Show statistics
print("\n=== Statistics ===")
print(f"Original dataset rows: {df.shape[0] - len(new_rows)}")
print(f"New Calculated National Average rows created: {len(new_rows)}")
print(f"Total rows now: {df.shape[0]}")

# Check if there are any existing National Average markets
existing_nat_avg = df[df['market'] == 'National Average']
print(f"\nExisting 'National Average' market rows (unchanged): {len(existing_nat_avg)}")
print(f"New 'Calculated National Average' rows (our calculation): {len(new_rows)}")

print(f"\nCalculated National Average price_per_unit stats:")
calc_nat_avg_values = df[df['market'] == 'Calculated National Average']['price_per_unit']
print(f"  Mean: {calc_nat_avg_values.mean():.4f}")
print(f"  Std: {calc_nat_avg_values.std():.4f}")
print(f"  Min: {calc_nat_avg_values.min():.4f}")
print(f"  Max: {calc_nat_avg_values.max():.4f}")


Reading CSV file...
Original data shape: (80680, 20)
Columns: ['date', 'price_usd', 'commodity_name', 'commodity_type', 'country', 'iso3', 'region', 'subregion', 'market', 'price_type', 'unit', 'unit_std', 'price_per_unit', 'price_source', 'gdp_ppp17', 'gdppc_ppp', 'gnipc_ppp', 'population', 'inflation', 'income_level']

Filtered to Domestic price_source: 77723 rows

Processing national averages...
Total groups to process: 31559
  Processed 1000/31559 groups...
  Processed 2000/31559 groups...
  Processed 3000/31559 groups...
  Processed 4000/31559 groups...
  Processed 5000/31559 groups...
  Processed 6000/31559 groups...
  Processed 7000/31559 groups...
  Processed 8000/31559 groups...
  Processed 9000/31559 groups...
  Processed 10000/31559 groups...
  Processed 11000/31559 groups...
  Processed 12000/31559 groups...
  Processed 13000/31559 groups...
  Processed 14000/31559 groups...
  Processed 15000/31559 groups...
  Processed 16000/31559 groups...
  Processed 17000/31559 groups..